In [1]:
import pandas as pd
import joblib
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

# 1. Load Data

In [2]:
df = pd.read_csv("../dataset/Crop_recommendation.csv")
df = df.drop(columns=[c for c in df.columns if c.lower().startswith('unnamed')])
df.columns = [c.lower() for c in df.columns]

In [3]:
X = df.drop('label', axis=1)
y = df['label']

# 2. Feature Importance Reporting (For Research Insights ONLY)

In [4]:
# We do not drop features, we just measure them for the research paper.
print("--- Baseline Feature Importance ---")

# Mutual Information
mi_scores = mutual_info_classif(X, y)
mi_df = pd.DataFrame({'feature': X.columns, 'mi_score': mi_scores})
print("\nMutual Information Scores:\n", mi_df.sort_values(by='mi_score', ascending=False))

--- Baseline Feature Importance ---

Mutual Information Scores:
        feature  mi_score
4     humidity  1.729954
2            k  1.653101
6     rainfall  1.637358
1            p  1.290612
3  temperature  1.017901
0            n  0.974436
5           ph  0.686067


# Random Forest Importance

In [5]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X, y)
fi_df = pd.DataFrame({'feature': X.columns, 'importance': rf.feature_importances_})
print("\nTree-based Importance:\n", fi_df.sort_values(by='importance', ascending=False))


Tree-based Importance:
        feature  importance
6     rainfall    0.226885
4     humidity    0.213955
2            k    0.180255
1            p    0.150093
0            n    0.102958
3  temperature    0.073629
5           ph    0.052225


# 3. Define Ablation Study Feature Sets

In [6]:
ablation_groups = {
    "all_features": list(X.columns),
    "soil_only": ['n', 'p', 'k', 'ph'],
    "climate_only": ['temperature', 'humidity', 'rainfall']
}

# 4. Save Artifacts for Modeling Phase

In [7]:
# Notice we save the unscaled X directly to preserve domain units for XAI
X.to_csv("../dataset/X_processed.csv", index=False)
y.to_csv("../dataset/y_processed.csv", index=False)
joblib.dump(ablation_groups, "../models/ablation_groups.pkl")

print("\nSaved unscaled data and ablation groups successfully.")


Saved unscaled data and ablation groups successfully.
